# Making a Salary Predictor

## Data 

### Loading the data

In [1]:
import pandas as pd

data = pd.read_csv('datasets/job_salary_prediction_dataset.csv')
data

,job_title,experience_years,education_level,skills_count,industry,company_size,location,remote_work,certifications,salary
0,AI Engineer,10,Bachelor,2,Healthcare,Medium,India,Hybrid,2,109413
1,Data Analyst,5,Bachelor,17,Telecom,Small,Australia,No,0,93764
2,Frontend Developer,18,PhD,4,Media,Medium,Singapore,No,1,148123
3,Business Analyst,19,PhD,13,Retail,Medium,Canada,Yes,0,189123
4,Product Manager,15,Bachelor,7,Manufacturing,Large,Sweden,Yes,0,165069
...,...,...,...,...,...,...,...,...,...,...
249995,Software Engineer,17,PhD,2,Telecom,Enterprise,India,No,1,127791
249996,Frontend Developer,20,PhD,7,Telecom,Startup,Remote,No,2,154593
249997,Business Analyst,1,Bachelor,12,Retail,Enterprise,India,Yes,0,75988
249998,Data Scientist,0,High School,2,Consulting,Small,Sweden,Hybrid,5,90467


### Data Analyst Average Salary

In [2]:
data['salary'][data['job_title']=='Data Analyst'].mean()

np.float64(119891.69660264453)

### Average Salary per Job Title

In [3]:
data['salary'].groupby(data['job_title']).mean()

job_title
AI Engineer                  173498.480640
Backend Developer            139202.768663
Business Analyst             122551.231354
Cloud Engineer               152102.535290
Cybersecurity Analyst        148697.695548
Data Analyst                 119891.696603
Data Scientist               147258.214409
DevOps Engineer              149959.266791
Frontend Developer           132653.842485
Machine Learning Engineer    163022.504570
Product Manager              157594.932029
Software Engineer            141739.521460
Name: salary, dtype: float64

### Average Salary per Education Level

In [4]:
data['salary'].groupby(data['education_level']).mean()

education_level
Bachelor       142410.531291
Diploma        137158.574976
High School    131715.336243
Master         153305.307833
PhD            163976.005295
Name: salary, dtype: float64

## Data Preparing

In [5]:
data

,job_title,experience_years,education_level,skills_count,industry,company_size,location,remote_work,certifications,salary
0,AI Engineer,10,Bachelor,2,Healthcare,Medium,India,Hybrid,2,109413
1,Data Analyst,5,Bachelor,17,Telecom,Small,Australia,No,0,93764
2,Frontend Developer,18,PhD,4,Media,Medium,Singapore,No,1,148123
3,Business Analyst,19,PhD,13,Retail,Medium,Canada,Yes,0,189123
4,Product Manager,15,Bachelor,7,Manufacturing,Large,Sweden,Yes,0,165069
...,...,...,...,...,...,...,...,...,...,...
249995,Software Engineer,17,PhD,2,Telecom,Enterprise,India,No,1,127791
249996,Frontend Developer,20,PhD,7,Telecom,Startup,Remote,No,2,154593
249997,Business Analyst,1,Bachelor,12,Retail,Enterprise,India,Yes,0,75988
249998,Data Scientist,0,High School,2,Consulting,Small,Sweden,Hybrid,5,90467


### Separate the data

In [6]:
# Separate features and labels
X = data.drop('salary', axis=1)
y = data['salary']

### Possible values for Education Level

In [7]:
data['education_level'].value_counts()

education_level
Master         50352
High School    50065
Bachelor       49950
PhD            49857
Diploma        49776
Name: count, dtype: int64

### Encoder and Scaler

In [8]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler

# Types of encoder
num_cols = ['experience_years', 'skills_count', 'certifications']
cat_hot_cols = ['job_title', 'company_size', 'location', 'remote_work']
cat_ord_cols = ['education_level']

# Order of Ordinal Encoder
ordinal_categories = [
    ['High School', 'Diploma', 'Bachelor', 'Master', 'PhD']
]

# Instance to Transform X
preprocessor = ColumnTransformer(
    transformers = [
        ('num', StandardScaler(), num_cols),
        ('cat_hot', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_hot_cols),
        ('cat_ord', OrdinalEncoder(categories=ordinal_categories), cat_ord_cols)
    ]
)

# Transform X
X_encoded = preprocessor.fit_transform(X)

### Split the Data

In [9]:
from sklearn.model_selection import train_test_split

train_data, test_data, train_labels, test_labels = train_test_split(
    X_encoded, y, train_size=0.8
)

### Pytorch Data Structures

In [10]:
# torch Datasets
import torch

# Tensors
X_train_tensor = torch.tensor(train_data, dtype=torch.float32)
y_train_tensor = torch.tensor(train_labels.to_numpy(), dtype=torch.float32)

X_test_tensor = torch.tensor(test_data, dtype=torch.float32)
y_test_tensor = torch.tensor(test_labels.to_numpy(), dtype=torch.float32)

# Datasets
from torch.utils.data import TensorDataset

train_data_dataset = TensorDataset(
    X_train_tensor,
    y_train_tensor
)

test_data_dataset = TensorDataset(
    X_test_tensor,
    y_test_tensor
)

# torch Dataloaders
from torch.utils.data import DataLoader

train_loader = DataLoader(train_data_dataset, shuffle=True, batch_size=16)
test_loader = DataLoader(test_data_dataset, shuffle=True, batch_size=16)

## Artificial Neural Network Model

In [11]:
import torch.nn as nn 

class SalaryPredictorModel(nn.Module):

    def __init__(self):
        super().__init__()

        self.model = nn.Sequential(
            nn.Linear(34, 64), # input
            nn.ReLU(),

            nn.Linear(64,32),
            nn.ReLU(),

            nn.Linear(32, 16),
            nn.ReLU(),

            nn.Linear(16, 1) # output

        )

    def forward(self, x):
        return self.model(x)

## Training

### Declaring the model and Tools

In [12]:
# Model
model = SalaryPredictorModel()

# Tools
learningrate = 0.001

optimizer = torch.optim.Adam(model.parameters(), lr=learningrate)

lossfun = nn.MSELoss()

### Training Loop

In [18]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

SalaryPredictorModel(
  (model): Sequential(
    (0): Linear(in_features=34, out_features=64, bias=True)
    (1): ReLU()
    (2): Linear(in_features=64, out_features=32, bias=True)
    (3): ReLU()
    (4): Linear(in_features=32, out_features=16, bias=True)
    (5): ReLU()
    (6): Linear(in_features=16, out_features=1, bias=True)
  )
)

In [ ]:
for epoch in range(num_epochs):

    # ===== Training =====
    model.train()
    epoch_loss = 0
    batchError = []

    for X, y in train_loader:
        X = X.to(device)
        y = y.float().view(-1,1).to(device)

        # Forward
        yHat = model(X)
        loss = lossfun(yHat, y)

        # Backward
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()

        # MAE
        mae = torch.abs(yHat - y).mean()
        batchError.append(mae.item())

    trainAcc.append(sum(batchError)/len(batchError))
    losses.append(epoch_loss/len(train_loader))

    # ===== Testing =====
    model.eval()
    batchError = []

    with torch.no_grad():
        for X, y in test_loader:
            X = X.to(device)
            y = y.float().view(-1,1).to(device)

            yHat = model(X)

            mae = torch.abs(yHat - y).mean()
            batchError.append(mae.item())

    testAcc.append(sum(batchError)/len(batchError))

    print(f"{epoch} | Train MAE: {trainAcc[-1]:.2f} | Test MAE: {testAcc[-1]:.2f}")

0 | Train MAE: 4560.86 | Test MAE: 4559.11
1 | Train MAE: 4556.80 | Test MAE: 4582.62
2 | Train MAE: 4546.61 | Test MAE: 4545.01
3 | Train MAE: 4540.20 | Test MAE: 4529.22
4 | Train MAE: 4532.42 | Test MAE: 4518.99
5 | Train MAE: 4528.19 | Test MAE: 4539.89
6 | Train MAE: 4520.06 | Test MAE: 4522.81
7 | Train MAE: 4514.62 | Test MAE: 4550.64
8 | Train MAE: 4411.41 | Test MAE: 4293.40
9 | Train MAE: 4264.84 | Test MAE: 4251.98
10 | Train MAE: 4243.27 | Test MAE: 4297.50
11 | Train MAE: 4232.81 | Test MAE: 4226.86
12 | Train MAE: 4228.19 | Test MAE: 4249.35
13 | Train MAE: 4223.49 | Test MAE: 4267.99
14 | Train MAE: 4223.77 | Test MAE: 4211.50
15 | Train MAE: 4222.74 | Test MAE: 4274.75
16 | Train MAE: 4220.59 | Test MAE: 4222.09
17 | Train MAE: 4220.51 | Test MAE: 4222.84
18 | Train MAE: 4219.34 | Test MAE: 4208.54
19 | Train MAE: 4219.44 | Test MAE: 4212.00
20 | Train MAE: 4219.85 | Test MAE: 4229.00
21 | Train MAE: 4216.28 | Test MAE: 4201.71
22 | Train MAE: 4216.31 | Test MAE: 4222.4